<h1>Custom feature extractor and preprocessing data </h1>

<p>Import libraries and combining all the fold data into one table</p>

In [20]:
import os
import numpy as np
import pandas as pd
import cv2
from skimage import feature
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from sklearn import svm



#data path and setting up data
data = r"C:\Users\keane\PycharmProjects\pythonProject\fyp\Adience\fold"
data_list = []
txt_files = os.listdir(data)

#loop through the folder and
for files in txt_files:
    count = int(files.split("_")[1])
    new = os.path.join(data, files)
    data_list.append(pd.read_table(new))

full_data = pd.concat(data_list, ignore_index=True)
print(full_data.shape)
print(full_data.head())

(19370, 12)
        user_id                original_image  face_id       age gender     x  \
0  30601258@N03  10399646885_67c7d20df9_o.jpg        1  (25, 32)      f     0   
1  30601258@N03  10424815813_e94629b1ec_o.jpg        2  (25, 32)      m   301   
2  30601258@N03  10437979845_5985be4b26_o.jpg        1  (25, 32)      f  2395   
3  30601258@N03  10437979845_5985be4b26_o.jpg        3  (25, 32)      m   752   
4  30601258@N03  11816644924_075c3d8d59_o.jpg        2  (25, 32)      m   175   

      y    dx    dy  tilt_ang  fiducial_yaw_angle  fiducial_score  
0   414  1086  1383      -115                  30              17  
1   105   640   641         0                   0              94  
2   876   771   771       175                 -30              74  
3  1255   484   485       180                   0              47  
4    80   769   768       -75                   0              34  


<p>Copying the age, gender and add it to the a new table with the file paths</p>

In [21]:
df = full_data[['age', 'gender']].copy()

img_path = []
for row in full_data.iterrows():
    path = r"C:/Users/keane/PycharmProjects/pythonProject/fyp/Adience/data/faces/" + row[1].user_id + "/coarse_tilt_aligned_face."+ str(row[1].face_id)+"."+ row[1].original_image
    img_path.append(path)
df['img_path'] = img_path
#df.head()

print(df.age.unique())
print(df.gender.unique())

['(25, 32)' '(38, 43)' '(4, 6)' '(60, 100)' '(15, 20)' '(48, 53)'
 '(8, 12)' '(0, 2)' 'None' '(38, 48)' '35' '3' '55' '58' '22' '13' '45'
 '36' '23' '(38, 42)' '(8, 23)' '(27, 32)' '57' '56' '2' '29' '34' '42'
 '46' '32']
['f' 'm' nan 'u']


<p>Cleaning all the none/NAN data from the table and removing u data from gender</p>

In [22]:
age_maps = [('(0, 2)', '0-2'), ('2', '0-2'), ('3', '0-2'), ('(4, 6)', '4-6'), ('(8, 12)', '8-13'), ('13', '8-13'), ('22', '15-20'), ('(8, 23)','15-20'), ('23', '25-32'), ('(15, 20)', '15-20'), ('(25, 32)', '25-32'), ('(27, 32)', '25-32'), ('32', '25-32'), ('34', '25-32'), ('29', '25-32'), ('(38, 42)', '38-43'), ('35', '38-43'), ('36', '38-43'), ('42', '48-53'), ('45', '38-43'), ('(38, 43)', '38-43'), ('(38, 42)', '38-43'), ('(38, 48)', '48-53'), ('46', '48-53'), ('(48, 53)', '48-53'), ('55', '48-53'), ('56', '48-53'), ('(60, 100)', '60+'), ('57', '60+'), ('58', '60+')]
gender = ['f','m','u']

age_maps_dict = {age[0]: age[1] for age in age_maps}
delete = []
for index, ages in enumerate(df.age):
    if ages == 'None':
        delete.append(index)
    else:
        df.age.loc[index] = age_maps_dict[ages]

df = df.drop(labels=delete, axis=0)
df = df.dropna()

clean_data = df[df.gender != 'u'].copy()
clean_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 17452 entries, 0 to 19345
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   age       17452 non-null  object
 1   gender    17452 non-null  object
 2   img_path  17452 non-null  object
dtypes: object(3)
memory usage: 545.4+ KB


<p>Convert the gender and age to labels</p>

In [23]:
gender_label = {'m' : 0,  'f' : 1}
age_label = {
    '0-2' : 0,
    '4-6' : 1,
    '8-13' : 2,
    '15-20' : 3,
    '25-32' : 4,
    '38-43' : 5,
    '48-53' : 6,
    '60+' : 7
}

clean_data['age'] = clean_data['age'].apply(lambda age: age_label[age])
clean_data['gender'] = clean_data['gender'].apply(lambda  gender: gender_label[gender])

In [24]:
clean_data.head()

,age,gender,img_path
0,4,1,C:/Users/keane/PycharmProjects/pythonProject/f...
1,4,0,C:/Users/keane/PycharmProjects/pythonProject/f...
2,4,1,C:/Users/keane/PycharmProjects/pythonProject/f...
3,4,0,C:/Users/keane/PycharmProjects/pythonProject/f...
4,4,0,C:/Users/keane/PycharmProjects/pythonProject/f...


In [39]:
features = []
labels = []
ages = []
gender = []

for row in clean_data.iterrows():
    image = cv2.imread(row[1].img_path)
    resized = resize(image, (128, 128))
    hog_feature = feature.hog(resized, channel_axis=-1)
    features.append(hog_feature)
    ages.append(row[1].age)
    gender.append(row[1].gender)


X = np.array(features)
age_labels = np.array(ages)
gender_labels = np.array(gender)



X_train, X_test, age_train, age_test, gender_train, gender_test = train_test_split(
    X, age_labels, gender_labels, test_size=0.1, random_state=10)

classifier_age = svm.SVC()
classifier_gender = svm.SVC()

classifier_age.fit(X_train, age_train)
classifier_gender.fit(X_train, gender_train)

SVC()

In [41]:
from sklearn.metrics import accuracy_score
age_pred = classifier_age.predict(X_test)
age_accuracy = accuracy_score(age_pred, age_test)

gender_pred = classifier_gender.predict(X_test)
gender_accuracy = accuracy_score(gender_pred, gender_test)

In [42]:
print(age_accuracy)
print(gender_accuracy)

0.5790378006872853
0.854524627720504


In [ ]:
image = cv2.imread(clean_data["img_path"][1])
grey_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#hog, hog_image = feature.hog(grey_image, orientations=9, pixels_per_cell=(16,16), cells_per_block=(1,1), visualize=True, block_norm='L2-Hys')
hog, hog_image = feature.hog(image, orientations=32, pixels_per_cell=(16,16), cells_per_block=(2,2), visualize=True, channel_axis=-1)
hog_feature= feature.hog(grey_image)
print(hog_feature.shape)
print(hog.shape)

#cv2.imshow('Grey', image)
#cv2.imshow('Image', hog_image)
#cv2.waitKey(0)

In [ ]:
# import random
# import os
# import cv2
# import numpy as np
# from sklearn.svm import SVC
#
#
# # Load the training data
# training_data = []
# training_labels = []
#
# # Data path and random seed
# random.seed(10)
# train = r"C:\Users\keane\PycharmProjects\pythonProject\testing\Adience\data\train"
# test = r"C:\Users\keane\PycharmProjects\pythonProject\testing\Adience\data\test"
#
# labels = os.listdir(train)
#
# for label in labels:
#     train_label = os.path.join(train, label)
#     test_label = os.path.join(test, label)
#     img_files = os.listdir(train_label)
#
#     #shuffle the files
#     random.shuffle(img_files)
#
#     for img in img_files:
#         full = os.path.join(train_label, img)
#         #image = cv2.imread("face_images/face_{}.jpg".format(full))
#         image = cv2.imread(full)
#         gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#         features = np.reshape(gray_image, (-1, 1))
#         training_data.append(features)
#         training_labels.append(img)
#
# # Train the classifier
# classifier = SVC(kernel="linear")
# classifier.fit(training_data, training_labels)
#
# # Test the classifier
# testimg_files = os.listdir(test_label)
# for testimg in testimg_files:
#     test_image = cv2.imread("face_images/face_{}.jpg".format(testimg))
#     gray_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2GRAY)
#     features = np.reshape(gray_image, (-1, 1))
#     prediction = classifier.predict(features)
#
# # Print the results
# print("Prediction:", prediction)


